In [296]:
!pip install iterative-stratification tqdm scikit-learn numpy torch transformers pandas scikit-multilear datetime

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: Could not find a version that satisfies the requirement scikit-multilear (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for scikit-multilear


In [297]:
import datetime
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F

start_time = datetime.datetime.now()
print(f"Script started at {start_time}")

timing_info = {}
timing_info['Start timing'] = start_time

Script started at 2026-02-25 11:18:59.134019


In [298]:
# Configuration parameters
two_emotions = False
file_name = "GroundTruth.csv"
checkpoint = "bert-base-cased"
num_folds = 5
emotions = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]

# Learning constraints
max_labels = 1

if two_emotions:
    file_name = "GroundTruthJoySadness.csv"
    emotions = ["Joy", "Sadness"]

# Undersampling configuration
undersample = True
cut_off = 150

# Class weight configuration
class_weight = True
function = 0  # options: BCEWithLogitsLoss (0), FocalLoss (1) or CrossEntropyLoss (2)
reduction = "mean" # options: mean or sum
gamma = 2.0  # for Focal Loss, when incrementing the value, focus more on hard examples.

In [299]:
# Configure output file path
suffixes = []
if undersample:
    suffixes.append("undersample")
if class_weight:
    suffixes.append("class_weight")
    if function == 0:
        suffixes.append("bce_with_logits_loss")
    elif function == 1:
        suffixes.append("focal_loss")
    elif function == 2:
        suffixes.append("cross_entropy_loss")

    suffixes.append(f"reduction_{reduction}")

prefix = "results_2e" if two_emotions else "results_all"

suffix_part = f"_{'_'.join(suffixes)}" if suffixes else ""

# Combine everything
output_path_csv = f"results/{prefix}_max{max_labels}_with_folds_{num_folds}{suffix_part}.csv"

In [300]:
df = pd.read_csv(file_name, quotechar='"', sep=';')
#df = df[df['Reject'] != 1]  # Filter out rows where Reject is 1

In [301]:
if undersample:
    start_undersampling = datetime.datetime.now()
    rng = np.random.default_rng(42)
    df = df.reset_index(drop=True)
    pos_counts = df[emotions].sum(axis=0).astype(int)
    targets = np.minimum(cut_off, pos_counts).astype(int)

    selected_idx = set()
    for lbl, tgt in zip(emotions, targets):
        if tgt <= 0:
            continue
        pos_idx = df.index[df[lbl] == 1].to_numpy()
        if len(pos_idx) <= tgt:
            chosen = pos_idx
        else:
            chosen = rng.choice(pos_idx, size=int(tgt), replace=False)
        selected_idx.update(int(x) for x in chosen)

    undersampled = df.loc[sorted(selected_idx)].reset_index(drop=True)
    print("Target per label:", dict(zip(emotions, targets.tolist())))
    print("Result per-label counts:\n", undersampled[emotions].sum().to_dict())
    print("Total rows selected:", len(undersampled))
    df = undersampled
    end_undersampling = datetime.datetime.now()
    timing_info['Undersampling time'] = end_undersampling - start_undersampling

Target per label: {'Joy': 150, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 150, 'Disgust': 50, 'Anger': 27, 'Anticipation': 150, 'Neutral': 88}
Result per-label counts:
 {'Joy': 191, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 180, 'Disgust': 50, 'Anger': 27, 'Anticipation': 155, 'Neutral': 88}
Total rows selected: 773


In [302]:
X = df[['sentence']].values
y = df[emotions].values

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [303]:
pos_weight = None

if class_weight:
    start_time_class_weight = datetime.datetime.now()
    num_pos = df[emotions].sum().values
    num_neg = len(df) - num_pos

    weights = num_neg / (num_pos + 1e-6)
    pos_weight = torch.tensor(weights, dtype=torch.float).to(device)

    print("Label counts (Positives):\n", num_pos)
    print("Computed pos_weights (Neg/Pos):\n", pos_weight)

    print("Weights per emotion:")
    for emotion, w in zip(emotions, pos_weight.tolist()):
        print(f"{emotion:12s}: {w:.3f}")

    timing_info['Class weight calculation time'] = datetime.datetime.now() - start_time_class_weight

In [304]:
class MultiLabelFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, reduction='mean'):
        super(MultiLabelFocalLoss, self).__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs,
            targets,
            reduction=reduction,
            pos_weight=self.pos_weight
        )

        pt = torch.exp(-bce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [305]:
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized

In [306]:
def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return batch_inputs

In [307]:
def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(emotions),
        problem_type="multi_label_classification"
    )
    model.to(device)
    return model

In [308]:
def get_constrained_predictions(logits):
    """
    Get top-k predictions based on max_labels constraint.

    Args:
        logits: tensor of shape (batch_size, num_labels)

    Returns:
        numpy array of shape (batch_size, num_labels) with 0/1 predictions
    """
    probs = torch.sigmoid(logits)

    # Get top max_labels indices per sample
    num_labels = probs.shape[1]
    k = min(max_labels, num_labels)

    _, top_indices = torch.topk(probs, k=k, dim=1)

    final_preds = torch.zeros_like(probs, dtype=torch.int)

    # Set predicted labels to 1
    for i, indices in enumerate(top_indices):
        final_preds[i, indices] = 1

    return final_preds.numpy()

In [309]:
multilabel_stratified_in_folds = MultilabelStratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
all_metrics = []
fold_results = []

In [310]:
for fold, (train_idx, val_idx) in enumerate(multilabel_stratified_in_folds.split(X, y)):
    timing_fold = datetime.datetime.now()
    print(f"\n===== Fold {fold + 1} =====")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = EmotionDataset(train_df, tokenizer, emotions)
    val_dataset = EmotionDataset(val_df, tokenizer, emotions)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

    model = create_model()

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 3
    num_training_steps = num_epochs * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # ---- training loop ----
    model.train()
    progress_bar = tqdm(range(num_training_steps))

    if class_weight and function == 0:
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 1:
        loss_fct = MultiLabelFocalLoss(gamma=2.0, pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 2:
        loss_fct = torch.nn.CrossEntropyLoss(weight=pos_weight, reduction=reduction)

    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            if class_weight:
                logits = outputs.logits
                loss = loss_fct(logits, batch["labels"].float())
            else:
                loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

    # ---- evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch["labels"].cpu().numpy().astype(int)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            preds = get_constrained_predictions(logits)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    sentences = val_df["sentence"].values


    def binvec_to_names(vec):
        names = [emotions[i] for i, v in enumerate(vec) if int(v) == 1]
        return ", ".join(names) if names else "None"


    true_emotions = [binvec_to_names(row) for row in y_true]
    pred_emotions = [binvec_to_names(row) for row in y_pred]

    df_predictions = pd.DataFrame({
        "sentence": sentences,
        "true_emotions": true_emotions,
        "pred_emotions": pred_emotions
    })

    # show a quick preview and optionally save per fold
    #print(df_predictions.head(20).to_string(index=False))
    df_predictions.to_csv(f"predictions/predictions_fold_{fold + 1}_out_of_{num_folds}.csv", index=False)

    fold_results.append({
        "fold": fold + 1,
        "y_true": y_true,
        "y_pred": y_pred
    })

    metrics = {
        "Subset Accuracy (Exact Match)": accuracy_score(y_true, y_pred),
        "Micro Precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro Recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro F1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Weighted Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

    # --- Print the Summary ---
    print(f"\n###### Metrics for fold {fold + 1} ######")
    for k, v in metrics.items():
        print(f"{k:25s}: {v:.4f}")

    all_metrics.append(metrics)
    end_timing_fold = datetime.datetime.now()
    timing_info[f'Fold {fold + 1} time'] = end_timing_fold - timing_fold


===== Fold 1 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 234/234 [05:36<00:00,  1.44s/it]

100%|██████████| 234/234 [02:34<00:00,  1.55it/s]


###### Metrics for fold 1 ######
Subset Accuracy (Exact Match): 0.3092
Micro Precision          : 0.4868
Micro Recall             : 0.4022
Micro F1                 : 0.4405
Macro Precision          : 0.3122
Macro Recall             : 0.2300
Macro F1                 : 0.1920
Weighted Precision       : 0.4978
Weighted Recall          : 0.4022
Weighted F1              : 0.3268

===== Fold 2 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 231/231 [02:54<00:00,  1.74it/s]


###### Metrics for fold 2 ######
Subset Accuracy (Exact Match): 0.4658
Micro Precision          : 0.5714
Micro Recall             : 0.5027
Micro F1                 : 0.5349
Macro Precision          : 0.3271
Macro Recall             : 0.3255
Macro F1                 : 0.3196
Weighted Precision       : 0.4794
Weighted Recall          : 0.5027
Weighted F1              : 0.4832

===== Fold 3 =====


...


###### Metrics for fold 3 ######
Subset Accuracy (Exact Match): 0.3831
Micro Precision          : 0.5000
Micro Recall             : 0.4162
Micro F1                 : 0.4543
Macro Precision          : 0.2483
Macro Recall             : 0.2450
Macro F1                 : 0.2296
Weighted Precision       : 0.4038
Weighted Recall          : 0.4162
Weighted F1              : 0.3823

===== Fold 4 =====


100%|██████████| 234/234 [02:25<00:00,  1.93it/s]Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



###### Metrics for fold 4 ######
Subset Accuracy (Exact Match): 0.2885
Micro Precision          : 0.4103
Micro Recall             : 0.3497
Micro F1                 : 0.3776
Macro Precision          : 0.2061
Macro Recall             : 0.2049
Macro F1                 : 0.1880
Weighted Precision       : 0.3351
Weighted Recall          : 0.3497
Weighted F1              : 0.3139

===== Fold 5 =====



100%|██████████| 234/234 [02:28<00:00,  1.58it/s]

100%|██████████| 234/234 [02:23<00:00,  1.87it/s]


###### Metrics for fold 5 ######
Subset Accuracy (Exact Match): 0.3667
Micro Precision          : 0.5533
Micro Recall             : 0.4511
Micro F1                 : 0.4970
Macro Precision          : 0.2612
Macro Recall             : 0.2763
Macro F1                 : 0.2597
Weighted Precision       : 0.4280
Weighted Recall          : 0.4511
Weighted F1              : 0.4240


In [311]:
print("\n###### Summary per fold ######")
timing_results = datetime.datetime.now()

df_metrics = pd.DataFrame(all_metrics).T  # transpose to have metrics as rows
df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
print("\n###### Overall Metrics per Fold ######")
print(df_metrics.to_markdown(floatfmt=".4f"))


###### Summary per fold ######

###### Overall Metrics per Fold ######
|                               |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:------------------------------|---------:|---------:|---------:|---------:|---------:|
| Subset Accuracy (Exact Match) |   0.3092 |   0.4658 |   0.3831 |   0.2885 |   0.3667 |
| Micro Precision               |   0.4868 |   0.5714 |   0.5000 |   0.4103 |   0.5533 |
| Micro Recall                  |   0.4022 |   0.5027 |   0.4162 |   0.3497 |   0.4511 |
| Micro F1                      |   0.4405 |   0.5349 |   0.4543 |   0.3776 |   0.4970 |
| Macro Precision               |   0.3122 |   0.3271 |   0.2483 |   0.2061 |   0.2612 |
| Macro Recall                  |   0.2300 |   0.3255 |   0.2450 |   0.2049 |   0.2763 |
| Macro F1                      |   0.1920 |   0.3196 |   0.2296 |   0.1880 |   0.2597 |
| Weighted Precision            |   0.4978 |   0.4794 |   0.4038 |   0.3351 |   0.4280 |
| Weighted Recall               |   0.

In [312]:
emotion_metrics = {emotion: {"Precision": [], "Recall": [], "F1": [], "Accuracy": []} for emotion in emotions}

for fold_data in fold_results:
    y_true = fold_data["y_true"]
    y_pred = fold_data["y_pred"]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    acc = [accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]

    for i, emotion in enumerate(emotions):
        emotion_metrics[emotion]["Precision"].append(prec[i])
        emotion_metrics[emotion]["Recall"].append(rec[i])
        emotion_metrics[emotion]["F1"].append(f1[i])
        emotion_metrics[emotion]["Accuracy"].append(acc[i])

In [313]:
print("\n###### Per Emotion Metrics per Fold ######")
for emotion, metrics_dict in emotion_metrics.items():
    df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
    print(f"\n--- {emotion} ---")
    print(df_emotion.to_markdown(floatfmt=".4f"))


###### Per Emotion Metrics per Fold ######

--- Joy ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.4559 |   0.6000 |   0.4898 |   0.3600 |   0.6000 |
| Recall    |   0.8158 |   0.5526 |   0.6154 |   0.4737 |   0.3947 |
| F1        |   0.5849 |   0.5753 |   0.5455 |   0.4091 |   0.4762 |
| Accuracy  |   0.7105 |   0.8075 |   0.7403 |   0.6667 |   0.7800 |

--- Trust ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.9091 |   0.4138 |   0.6500 |   0.6111 |   0.5789 |
| Recall    |   0.3333 |   0.4000 |   0.4333 |   0.3667 |   0.7333 |
| F1        |   0.4878 |   0.4068 |   0.5200 |   0.4583 |   0.6471 |
| Accuracy  |   0.8618 |   0.7826 |   0.8442 |   0.8333 |   0.8400 |

--- Fear ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|----

In [314]:
print("\n###### Average Per-Emotion Metrics ######")
df_emotion_avg = pd.DataFrame({
    emotion: {metric: np.mean(values) for metric, values in metrics.items()}
    for emotion, metrics in emotion_metrics.items()
})
print(df_emotion_avg.T.to_markdown(floatfmt=".4f"))


###### Average Per-Emotion Metrics ######
|              |   Precision |   Recall |     F1 |   Accuracy |
|:-------------|------------:|---------:|-------:|-----------:|
| Joy          |      0.5011 |   0.5704 | 0.5182 |     0.7410 |
| Trust        |      0.6326 |   0.4533 | 0.5040 |     0.8324 |
| Fear         |      0.0000 |   0.0000 | 0.0000 |     0.9690 |
| Surprise     |      0.0000 |   0.0000 | 0.0000 |     0.9301 |
| Sadness      |      0.4416 |   0.7944 | 0.5650 |     0.7137 |
| Disgust      |      0.0000 |   0.0000 | 0.0000 |     0.9353 |
| Anger        |      0.0000 |   0.0000 | 0.0000 |     0.9650 |
| Anticipation |      0.7233 |   0.4065 | 0.4491 |     0.8430 |
| Neutral      |      0.1400 |   0.0824 | 0.1037 |     0.8897 |


In [315]:
df_metrics = pd.DataFrame(all_metrics)
print("\n###### Cross-validation mean results ######")
print(df_metrics.mean())

timing_info[f'Printing results'] = datetime.datetime.now() - timing_results

with open(output_path_csv, mode="w", newline="") as f:
    writer = csv.writer(f, delimiter=";")

    # ===== SUMMARY PER FOLD ######
    writer.writerow(["###### Summary per fold ######"])
    df_metrics = pd.DataFrame(all_metrics).T  # metrics as rows
    df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
    df_metrics.reset_index(inplace=True)
    df_metrics.rename(columns={"index": "Metric"}, inplace=True)
    df_metrics.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
    writer.writerow([])  # blank line

    # ###### PER-EMOTION METRICS PER FOLD ######
    writer.writerow(["###### Per Emotion Metrics per Fold ######"])
    for emotion, metrics_dict in emotion_metrics.items():
        writer.writerow([f"--- {emotion} ---"])
        df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
        df_emotion.reset_index(inplace=True)
        df_emotion.rename(columns={"index": "Metric"}, inplace=True)
        df_emotion.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
        writer.writerow([])
        writer.writerow([])  # blank line

    # ###### AVERAGE PER EMOTION ######
    writer.writerow(["###### Average Per-Emotion Metrics ######"])
    df_emotion_avg = pd.DataFrame({
        emotion: {metric: np.mean(values) for metric, values in metrics.items()}
        for emotion, metrics in emotion_metrics.items()
    })
    df_emotion_avg = df_emotion_avg.T.reset_index().rename(columns={"index": "Emotion"})
    df_emotion_avg.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
    writer.writerow([])

    # ###### CROSS-VALIDATION MEAN RESULTS ######
    writer.writerow(["###### Cross-validation Mean Results ######"])
    df_metrics = pd.DataFrame(all_metrics)
    df_mean = df_metrics.mean().to_frame(name="Mean").reset_index().rename(columns={"index": "Metric"})
    df_mean.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")

    # ###### CLASS WEIGHTS ######
    if class_weight:
        writer.writerow([])
        writer.writerow(["###### Class Weights per Emotion ######"])
        writer.writerow(["Emotion", "Weight"])
        for emotion, weight in zip(emotions, pos_weight.tolist()):
            writer.writerow([emotion, f"{weight:.4f}"])
        writer.writerow([])

    writer.writerow(["###### Timings ######"])
    for k, v in timing_info.items():
        writer.writerow([k, str(v)])

    writer.writerow(["Total duration", str(datetime.datetime.now() - start_time)])

print(f"\n✅ All tables saved in one CSV file: {os.path.abspath(output_path_csv)}")


###### Cross-validation mean results ######
Subset Accuracy (Exact Match)    0.362659
Micro Precision                  0.504372
Micro Recall                     0.424387
Micro F1                         0.460845
Macro Precision                  0.270960
Macro Recall                     0.256336
Macro F1                         0.237779
Weighted Precision               0.428813
Weighted Recall                  0.424387
Weighted F1                      0.386028
dtype: float64

✅ All tables saved in one CSV file: /Users/carlotacatot/Docs/pHd/EmotionClassification/results/results_all_max1_with_folds_5_undersample.csv
